# 🧪 W3-D4 概念实验：DPO vs RLHF 收敛对比 & 对齐方法全景

> 配套阅读：`ima/第3周-Day4-DPO与其他对齐方法对比.md`
>
> DPO 革命：把 RLHF 简化为一个损失函数。

## 实验 1：DPO Loss 函数直觉

让好回答概率上升、差回答概率下降。

In [ ]:
import numpy as np

def dpo_loss(lp_w, lp_l, lp_w_ref, lp_l_ref, beta=0.1):
    r_w = beta * (lp_w - lp_w_ref)
    r_l = beta * (lp_l - lp_l_ref)
    logit = r_w - r_l
    return -np.log(1 / (1 + np.exp(-logit)))

cases = [("模型正确区分", -2.0, -4.0, -3.0, -3.0),
         ("模型判断错误", -4.0, -2.0, -3.0, -3.0),
         ("模型还没学会", -3.0, -3.0, -3.0, -3.0)]
for name, pw, pl, rw, rl in cases:
    print(f"  {name:12s} → Loss = {dpo_loss(pw, pl, rw, rl):.4f}")
print("→ 已区分好坏时 Loss 最低；判断错误时最高。")

## 实验 2：DPO vs RLHF 训练稳定性

RLHF 有 PPO 震荡，DPO 稳定下降。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(4); epochs = 60
erlhf = 2.5*np.exp(-np.arange(epochs)/15)+0.3+0.15*np.sin(np.arange(epochs)/3)+np.random.normal(0,0.08,epochs)
erlhf = np.maximum(erlhf, 0.2)
edpo = 2.5*np.exp(-np.arange(epochs)/12)+0.3+np.random.normal(0,0.03,epochs)
edpo = np.maximum(edpo, 0.2)
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(range(epochs), erlhf, 'r-', lw=1.5, label='RLHF (PPO)', alpha=0.8)
ax.plot(range(epochs), edpo, 'b-', lw=2, label='DPO（稳定下降）')
ax.set_xlabel('Epoch'); ax.set_ylabel('训练 Loss')
ax.set_title('DPO vs RLHF：DPO 曲线平稳')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 实验 3：β 参数对 DPO 的影响

β 太小→过度优化，太大→不学习。0.1~0.5 是甜蜜区。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
betas = np.linspace(0.01, 1.5, 150)
def dl(b, r): return -np.log(1/(1+np.exp(-b*r)))
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(betas, dl(betas,  2.0), 'g-', lw=2, label='模型已学会')
ax.plot(betas, dl(betas,  0.0), 'y-', lw=2, label='模型还没学会')
ax.plot(betas, dl(betas, -2.0), 'r-', lw=2, label='模型判断错误')
ax.axvspan(0.1, 0.5, alpha=0.1, color='blue', label='推荐β范围')
ax.set_xlabel('β'); ax.set_ylabel('DPO Loss')
ax.set_title('β 太小→Loss 接近0；β太大→无梯度')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 实验 4：对齐方法全景雷达图

RLHF、DPO、KTO、GRPO 六维对比。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
cats = ['训练稳定','资源友好','实现简单','效果上限','数据效率','调试容易']
rlhf,dpo,kto,grpo = [3,2,2,9,4,2],[8,7,8,7,6,8],[7,8,7,6,9,7],[6,6,5,8,8,5]
angles = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist()+[0]
fig, ax = plt.subplots(figsize=(7.5,7.5), subplot_kw=dict(polar=True))
for s,name,col in [(rlhf,'RLHF','red'),(dpo,'DPO','blue'),(kto,'KTO','green'),(grpo,'GRPO','orange')]:
    ax.plot(angles, s+s[:1], '-o', lw=2, label=name, color=col, ms=4)
    ax.fill(angles, s+s[:1], alpha=0.08, color=col)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats, fontsize=11)
ax.set_title('对齐方法四维对比', fontsize=14, pad=20)
ax.legend(loc='lower right', fontsize=10); plt.tight_layout(); plt.show()

## 结论

| 问题 | 实验证据 |
|---|---|
| DPO Loss | 实验1：好回答概率上升=最小化 Loss |
| 稳定性 | 实验2：DPO 稳定，RLHF 震荡 |
| β 选择 | 实验3：0.1~0.5 甜蜜区 |
| 选型 | 实验4：DPO 首选，KTO 省标注，GRPO 提推理 |

→ 配套阅读：`ima/第3周-Day4-DPO与其他对齐方法对比.md`